In [6]:
import ee
import requests
from pathlib import Path
import rioxarray as rxr

ee.Initialize()

# 1) Build the image you want (e.g., first image in date range)
ic = (ee.ImageCollection("MODIS/061/MCD43A3")
      .filterDate("2018-01-01", "2019-05-01")
      .select("Albedo_BSA_Band1"))

img = ee.Image(ic.first())

# 2) Define a small region (example: a box in lon/lat)
region = ee.Geometry.Rectangle([0, -75, 10, -70])  # [xmin, ymin, xmax, ymax]

# 3) Get a GeoTIFF download URL (note: region must be small-ish)
url = img.getDownloadURL({
    "scale": 500,
    "region": region,
    "crs": "EPSG:4326",
    "format": "GEO_TIFF"
})

# 4) Download
out = Path("mcd43a3_bsa_band1_2018-01-01.tif")
r = requests.get(url, stream=True)
r.raise_for_status()
with out.open("wb") as f:
    for chunk in r.iter_content(chunk_size=1 << 20):
        f.write(chunk)

# 5) Open in xarray
da = rxr.open_rasterio(out).squeeze("band", drop=True)
print(da)

<xarray.DataArray (y: 1127, x: 2227)> Size: 5MB
[2509829 values with dtype=int16]
Coordinates:
  * x            (x) float64 18kB 0.002246 0.006737 0.01123 ... 9.992 9.996 10.0
  * y            (y) float64 9kB -70.0 -70.0 -70.01 ... -75.05 -75.05 -75.06
    spatial_ref  int64 8B 0
Attributes:
    TIFFTAG_XRESOLUTION:     1
    TIFFTAG_YRESOLUTION:     1
    TIFFTAG_RESOLUTIONUNIT:  1 (unitless)
    AREA_OR_POINT:           Area
    _FillValue:              -32768
    scale_factor:            1.0
    add_offset:              0.0


In [10]:
ic.getInfo()

{'type': 'ImageCollection',
 'bands': [],
 'version': 1772101866775708,
 'id': 'MODIS/061/MCD43A3',
 'properties': {'system:is_global': 1},
 'features': [{'type': 'Image',
   'bands': [{'id': 'Albedo_BSA_Band1',
     'data_type': {'type': 'PixelType',
      'precision': 'int',
      'min': -32768,
      'max': 32767},
     'dimensions': [86400, 38400],
     'crs': 'SR-ORG:6974',
     'crs_transform': [463.3127165279165,
      0,
      -20015109.354,
      0,
      -463.3127165279167,
      7783653.637669001]}],
   'version': 1766784912217728,
   'id': 'MODIS/061/MCD43A3/2018_01_01',
   'properties': {'system:time_start': 1514764800000,
    'google:max_source_file_timestamp': 1656304707000,
    'num_tiles': 299,
    'system:footprint': {'type': 'LinearRing',
     'coordinates': [[-180, -90],
      [180, -90],
      [180, 90],
      [-180, 90],
      [-180, -90]]},
    'system:time_end': 1514851200000,
    'system:asset_size': 16610344880,
    'system:index': '2018_01_01'}},
  {'type': '

In [11]:
len(ic)

TypeError: object of type 'ImageCollection' has no len()